# R14-H144 - the boundary audit: what the window severs

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-07 <br>
**Pipeline stage**: R14 ingest-fidelity round, boundary audit <br>
**Graph**: rebuilt CPAP corpus (neo4j2, read-only) - entity->document provenance only <br>

H51 measured parser-level loss; this measures chunker-level context severance. The 2000/200
token window with sentence snapping preserves strings but not associations - a table row landing
in a different chunk than its header loses the header binding even though every character survives.

## Approach (deterministic, as registered)
1. **Parse** all corpus PDFs with `pymupdf4llm.to_markdown` (the project reader), build `Document`s
2. **Chunk** each doc with `chunk_document` (2000/200, sentence snap) - the production chunker, re-run
   with char-offset instrumentation so every chunk maps to an exact `[start,end)` span of raw text
3. **(a) Entity-name severance** - entity names whose `source_documents` include the doc (from neo4j2);
   a name occurrence is SEVERED if it is not fully contained in any single chunk (the cut falls inside
   the name), and OVERLAP-ONLY if every occurrence lives inside a chunk-overlap seam
4. **(b) Table-row severance** - markdown table detection (header row + `---` separator); a data row is
   SEVERED when the chunk containing it does not also contain its header row
5. **Verdict** - document-level incidence of either effect: >= 10% confirms, < 2% refutes

## Outputs
- `reports/boundary-audit-h144-<stamp>.json`


In [1]:
# Imports
# stdlib
import datetime, json, os, re
from pathlib import Path
# third party
import tiktoken
from neo4j import GraphDatabase
from rich import print as rprint
from rich.progress import Progress
# project
from knowledge_graph_foundry.ingest.readers import read_document, _document_id
from knowledge_graph_foundry.ingest.chunking import chunk_document, _snap_to_boundary
from knowledge_graph_foundry.models import Document

NEO4J_URI = "bolt://user-konrad.jelen-kgf-neo4j2:7687"   # read-only neo4j2 (NOT .env live graph)
DOC_DIR = Path("../data/external/cpap-datasheets-and-manuals")
CHUNK_SIZE, CHUNK_OVERLAP = 2000, 200
BAR_CONFIRM, BAR_REFUTE = 0.10, 0.02
rprint(f"[bold]config[/bold] {CHUNK_SIZE}/{CHUNK_OVERLAP} snap  bar confirm>={BAR_CONFIRM} refute<{BAR_REFUTE}")


2026-07-07 12:13:27.449 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


config 2000/200 snap  bar confirm>=0.1 refute<0.02

## Parse corpus and instrument the chunker\n\nRe-implements `chunk_document`'s exact loop (verified against the shipped function output) while capturing each chunk's token-start, so every chunk maps to a precise char span in the raw text via `len(decode(tokens[:start]))`.

In [2]:
enc = tiktoken.get_encoding("cl100k_base")

def chunks_with_spans(doc):
    """chunk_document logic (chunking.py) + char-span capture. Asserts text-parity
    with the shipped chunk_document so the audit runs on the real chunks."""
    text = doc.text
    if not text.strip():
        return []
    tokens = enc.encode(text)
    total = len(tokens)
    out = []
    if total <= CHUNK_SIZE:
        out.append({"index": 0, "text": text, "cstart": 0, "cend": len(text)})
        return out
    start = index = 0
    while start < total:
        end = min(start + CHUNK_SIZE, total)
        piece = enc.decode(tokens[start:end])
        if end < total:
            piece = _snap_to_boundary(piece, enc, CHUNK_SIZE)
        if not piece.strip():
            start = end
            continue
        token_count = len(enc.encode(piece))
        cstart = len(enc.decode(tokens[:start]))
        out.append({"index": index, "text": piece, "cstart": cstart, "cend": cstart + len(piece)})
        index += 1
        if end >= total:
            break
        start += max(token_count - CHUNK_OVERLAP, 1)
    return out

docs = []
pdfs = sorted(DOC_DIR.glob("*.pdf"))
with Progress() as pr:
    t = pr.add_task("parse", total=len(pdfs))
    for p in pdfs:
        try:
            d = read_document(p)
        except Exception as e:
            rprint(f"[red]parse fail[/red] {p.name}: {e}")
            pr.advance(t); continue
        docid = _document_id(p)
        ch = chunks_with_spans(d)
        # parity check against shipped chunker
        shipped = [c.text for c in chunk_document(d, CHUNK_SIZE, CHUNK_OVERLAP)]
        parity = [c["text"] for c in ch] == shipped
        docs.append({"name": p.name, "docid": docid, "text": d.text, "chunks": ch,
                     "n_chunks": len(ch), "parity": parity})
        pr.advance(t)

n_parity = sum(d["parity"] for d in docs)
rprint(f"parsed [yellow]{len(docs)}[/yellow] docs  chunk-parity [green]{n_parity}/{len(docs)}[/green]  "
       f"multi-chunk docs [yellow]{sum(1 for d in docs if d['n_chunks']>1)}[/yellow]")


/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

2026-07-07 12:13:29.236 | DEBUG    | knowledge_graph_foundry.ingest.readers:read_document:68 - read 
0-20190113114505.pdf (4881 chars)

2026-07-07 12:13:33.241 | DEBUG    | knowledge_graph_foundry.ingest.readers:read_document:68 - read 
1017900r4_ResMed_Product_Catalogue_ANZ_Eng_LowRes.pdf (14762 chars)

2026-07-07 12:13:33.254 | DEBUG    | knowledge_graph_foundry.ingest.chunking:chunk_document:65 - chunked 
d_c544d2031f25c004 into 2 chunks

2026-07-07 12:13:44.265 | DEBUG    | knowledge_graph_foundry.ingest.readers:read_document:68 - read 
3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf (52638 chars)

...

2026-07-07 12:16:15.592 | DEBUG    | knowledge_graph_foundry.ingest.readers:read_document:68 - read 
moh-adp-product-manual-respiratory-devices-airway-clearance-en-2023-06-14.pdf (7326 chars)

2026-07-07 12:16:15.600 | DEBUG    | knowledge_graph_foundry.ingest.chunking:chunk_document:65 - chunked 
d_160ebdb9b2e36f50 into 2 chunks

2026-07-07 12:17:07.202 | DEBUG    | knowledge_graph_foundry.ingest.readers:read_document:68 - read 
product_and_solutions_catalog.pdf (210634 chars)

2026-07-07 12:17:07.384 | DEBUG    | knowledge_graph_foundry.ingest.chunking:chunk_document:65 - chunked 
d_2b3d647422cbe53e into 29 chunks

parsed 28 docs  chunk-parity 28/28  multi-chunk docs 17

## Entity-name provenance (neo4j2, read-only)\n\nFor each document id, the names of entities whose `source_documents` include it. These are the strings whose boundary integrity we audit.

In [3]:
drv = GraphDatabase.driver(NEO4J_URI, auth=("neo4j", "kgfoundry"), notifications_min_severity="OFF")
with drv.session() as s:
    ent_rows = s.run(
        "MATCH (e:Entity) WHERE e.source_documents IS NOT NULL "
        "RETURN e.name AS name, e.source_documents AS docs"
    ).data()
drv.close()

doc_entities = {}   # docid -> set of names
for r in ent_rows:
    nm = (r["name"] or "").strip()
    if len(nm) < 3:
        continue
    for did in r["docs"] or []:
        doc_entities.setdefault(did, set()).add(nm)

covered = sum(1 for d in docs if d["docid"] in doc_entities)
rprint(f"entities loaded [yellow]{len(ent_rows)}[/yellow]  docs with entity provenance [yellow]{covered}/{len(docs)}[/yellow]")


entities loaded 2798  docs with entity provenance 27/28

## (a) Entity-name severance\n\nA name occurrence (char span in raw text) is SEVERED if no single chunk fully contains it - the window cut falls inside the name. OVERLAP-ONLY if every occurrence of the name sits inside a chunk-overlap seam (present only because the 200-token overlap duplicates the seam).

In [4]:
def find_occurrences(needle, hay):
    """all char spans of needle in hay (case-insensitive, non-overlapping)."""
    spans = []
    n = needle.lower(); h = hay.lower()
    i = h.find(n)
    while i != -1:
        spans.append((i, i + len(n)))
        i = h.find(n, i + max(1, len(n)))
    return spans

def overlap_regions(chunks):
    """char spans shared by adjacent chunks (the 200-token seams)."""
    regs = []
    for a, b in zip(chunks, chunks[1:]):
        lo, hi = b["cstart"], a["cend"]
        if hi > lo:
            regs.append((lo, hi))
    return regs

name_audit = []
for d in docs:
    ch = d["chunks"]
    if len(ch) < 2:
        name_audit.append({"name": d["name"], "docid": d["docid"], "n_names": 0,
                           "severed": 0, "overlap_only": 0, "flag": False, "examples": []})
        continue
    names = doc_entities.get(d["docid"], set())
    regs = overlap_regions(ch)
    severed = ov_only = 0
    examples = []
    for nm in names:
        occ = find_occurrences(nm, d["text"])
        if not occ:
            continue
        # SEVERED: an occurrence contained in no single chunk
        nm_sev = any(not any(c["cstart"] <= a and b <= c["cend"] for c in ch) for a, b in occ)
        # OVERLAP-ONLY: every occurrence lies inside an overlap seam
        def in_overlap(a, b):
            return any(lo <= a and b <= hi for lo, hi in regs)
        nm_ovonly = bool(occ) and all(in_overlap(a, b) for a, b in occ)
        if nm_sev:
            severed += 1
            if len(examples) < 5:
                examples.append({"name": nm, "kind": "severed"})
        elif nm_ovonly:
            ov_only += 1
            if len(examples) < 5:
                examples.append({"name": nm, "kind": "overlap_only"})
    name_audit.append({"name": d["name"], "docid": d["docid"], "n_names": len(names),
                       "severed": severed, "overlap_only": ov_only,
                       "flag": (severed + ov_only) > 0, "examples": examples})

tot_sev = sum(a["severed"] for a in name_audit)
tot_ov = sum(a["overlap_only"] for a in name_audit)
docs_name_flag = sum(1 for a in name_audit if a["flag"])
rprint(f"[bold]name severance[/bold] severed occurrences={tot_sev}  overlap-only names={tot_ov}  "
       f"docs flagged [yellow]{docs_name_flag}/{len(docs)}[/yellow]")
for a in name_audit:
    if a["flag"]:
        rprint(f"  [dim]{a['name'][:45]:45s}[/dim] sev={a['severed']} ovonly={a['overlap_only']} "
               f"{[e['name'][:30] for e in a['examples'][:3]]}")


name severance severed occurrences=0  overlap-only names=204  docs flagged 16/28

1017900r4_ResMed_Product_Catalogue_ANZ_Eng_Lo sev=0 ovonly=4 ['Quattro Air for Her', 'Quattro Air', 'AirFit F10']

3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1. sev=0 ovonly=4 ['Max APAP', 'Full face mask', 'LCD Light']

ARTP_Standards_of_Care_-_CPAP_Devices_(Techni sev=0 ovonly=6 ['Adjustable Delay/Ramp', 'Unintentional Leak 
Compensatio', 'User Mode']

BMC_RESmart_AutoCPAP_User_Manual.pdf          sev=0 ovonly=1 ['Patient Disconnect Alert']

...

Seattle-PAP-V5-en-in.pdf                      sev=0 ovonly=2 ['Germany', 'Lübeck']

Sleep And Respiratory Medical Devices Brochur sev=0 ovonly=38 ['Allergies and hayfever', 'Transcend P10 battery',
'Spare Reusable Canister']

SleepStyle_200_Operating_Manual.pdf           sev=0 ovonly=6 ['HC230-Series', 'Continuous Positive Airway Pre', 
'Sleep Lab']

moh-adp-product-manual-respiratory-devices-ai sev=0 ovonly=1 ['Drive Medical']

product_and_solutions_catalog.pdf             sev=0 ovonly=88 ['Actiwatch 2 docking station', 'P1652', 'MPV 
circuit support system']

## (b) Table-row severance\n\nMarkdown table = a header line `|...|` immediately followed by a separator `|---|---|`. Each data row is SEVERED when the chunk containing it does not also contain its header row - the row lands in a different window than its column headings.

In [5]:
SEP_RE = re.compile(r"^\s*\|?\s*:?-{2,}:?\s*(\|\s*:?-{2,}:?\s*)+\|?\s*$")
ROW_RE = re.compile(r"^\s*\|.*\|\s*$")

def line_spans(text):
    """(line_text, char_start, char_end) for every line."""
    out = []
    pos = 0
    for ln in text.splitlines(keepends=True):
        out.append((ln.rstrip("\n"), pos, pos + len(ln.rstrip("\n"))))
        pos += len(ln)
    return out

def tables(text):
    """list of {header:(a,b), rows:[(a,b),...]} markdown tables."""
    ls = line_spans(text)
    out = []
    i = 0
    while i < len(ls) - 1:
        htxt, ha, hb = ls[i]
        stxt, _, _ = ls[i + 1]
        if ROW_RE.match(htxt) and SEP_RE.match(stxt):
            rows = []
            j = i + 2
            while j < len(ls) and ROW_RE.match(ls[j][0]):
                rows.append((ls[j][1], ls[j][2]))
                j += 1
            out.append({"header": (ha, hb), "rows": rows})
            i = j
        else:
            i += 1
    return out

def chunk_of(span, ch):
    """indices of chunks fully containing span (may be >1 due to overlap)."""
    a, b = span
    return [c["index"] for c in ch if c["cstart"] <= a and b <= c["cend"]]

table_audit = []
for d in docs:
    ch = d["chunks"]
    tbs = tables(d["text"])
    n_rows = sum(len(t["rows"]) for t in tbs)
    severed = 0
    examples = []
    if len(ch) >= 2:
        for t in tbs:
            hchunks = set(chunk_of(t["header"], ch))
            for r in t["rows"]:
                rchunks = set(chunk_of(r, ch))
                # severed: the row is contained in some chunk, but no chunk contains both row and header
                if rchunks and not (rchunks & hchunks):
                    severed += 1
                    if len(examples) < 5:
                        examples.append(d["text"][r[0]:r[1]][:70])
    table_audit.append({"name": d["name"], "n_tables": len(tbs), "n_rows": n_rows,
                        "severed_rows": severed, "flag": severed > 0, "examples": examples})

tot_tbl = sum(a["severed_rows"] for a in table_audit)
docs_tbl_flag = sum(1 for a in table_audit if a["flag"])
rprint(f"[bold]table severance[/bold] tables={sum(a['n_tables'] for a in table_audit)} "
       f"rows={sum(a['n_rows'] for a in table_audit)}  severed rows={tot_tbl}  "
       f"docs flagged [yellow]{docs_tbl_flag}/{len(docs)}[/yellow]")
for a in table_audit:
    if a["flag"]:
        rprint(f"  [dim]{a['name'][:45]:45s}[/dim] severed={a['severed_rows']}/{a['n_rows']} "
               f"{[e[:35] for e in a['examples'][:2]]}")


table severance tables=256 rows=2302  severed rows=64  docs flagged 7/28

3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1. severed=3/24 ['|Voltage dips,<br>short<br>interrup', 
'|Power<br>frequency<br>(50/60 Hz)<b']

BMC_RESmart_AutoCPAP_User_Manual.pdf          severed=3/61 ['|**Adjusting the Circuit**|2. Adjus', '|**Using the 
Ramp Button  **|Pressi']

DreamStation_CPAP_User_Manual.pdf             severed=13/138 ['||Serial number<br>Identify the man', 
'|**Periodic**<br>**Breathing**|Peri']

PDF RESmart Service Manual CPAP.pdf           severed=30/114 ['|Foam (cover)|40191-0201|1|PC|Foam|', '|Shied 
Cover|4011110101|1|PC|Plasti']

ResMed-Airsense-11-Manual.pdf                 severed=8/138 ['||The wireless signal strength icon', '|Device may 
be in Airplane Mode.|Tu']

Resvent-iBreeze-Auto-CPAP-User-Manual.pdf     severed=4/146 ['|High Respiratory<br>Rate (RR).|Ale', '|Low 
Respiratory<br>Rate (RR).|Aler']

moh-adp-product-manual-respiratory-devices-ai severed=3/50 ['||Medela<br>Healthcare|Clario Home ', '|||Clario 
Toni Home<br>Care Pump, A']

## Verdict\n\nDocument-level combined incidence: a document is flagged if it has any severed/overlap-only entity name OR any severed table row. Bar reads on the fraction of documents flagged.

In [6]:
flagged = set()
for a in name_audit:
    if a["flag"]:
        flagged.add(a["name"])
for a in table_audit:
    if a["flag"]:
        flagged.add(a["name"])

n_docs = len(docs)
incidence = len(flagged) / n_docs if n_docs else 0.0
verdict = ("CONFIRMED" if incidence >= BAR_CONFIRM else
           "REFUTED" if incidence < BAR_REFUTE else "INCONCLUSIVE")

rprint(f"""[bold cyan]Boundary audit - verdict[/bold cyan]
[dim]{"-"*44}[/dim]
  Documents: [yellow]{n_docs}[/yellow]  (multi-chunk: [yellow]{sum(1 for d in docs if d['n_chunks']>1)}[/yellow])
  Name-severance flagged docs: [yellow]{sum(1 for a in name_audit if a['flag'])}[/yellow]
  Table-severance flagged docs: [yellow]{sum(1 for a in table_audit if a['flag'])}[/yellow]
  Combined flagged docs: [yellow]{len(flagged)}[/yellow]
  Incidence: [bold yellow]{incidence:.3f}[/bold yellow] [dim](confirm >= {BAR_CONFIRM}, refute < {BAR_REFUTE})[/dim]
  Verdict: [{'green' if verdict=='CONFIRMED' else 'red' if verdict=='REFUTED' else 'yellow'}]{verdict}[/]
""")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
out = Path("../reports") / f"boundary-audit-h144-{stamp}.json"
out.write_text(json.dumps({
    "hypothesis": "R14-H144", "chunker": f"{CHUNK_SIZE}/{CHUNK_OVERLAP} sentence-snap",
    "n_documents": n_docs, "multi_chunk_docs": sum(1 for d in docs if d["n_chunks"] > 1),
    "total_severed_name_occurrences": tot_sev, "total_overlap_only_names": tot_ov,
    "total_severed_table_rows": tot_tbl,
    "docs_name_flagged": sum(1 for a in name_audit if a["flag"]),
    "docs_table_flagged": sum(1 for a in table_audit if a["flag"]),
    "combined_flagged_docs": sorted(flagged),
    "incidence": incidence, "bar_confirm": BAR_CONFIRM, "bar_refute": BAR_REFUTE,
    "verdict": verdict,
    "name_audit": name_audit, "table_audit": table_audit,
}, indent=2, default=str))
rprint("saved", str(out))


Boundary audit - verdict
--------------------------------------------
  Documents: 28  (multi-chunk: 17)
  Name-severance flagged docs: 16
  Table-severance flagged docs: 7
  Combined flagged docs: 17
  Incidence: 0.607 (confirm >= 0.1, refute < 0.02)
  Verdict: CONFIRMED

saved ../reports/boundary-audit-h144-20260707-101708.json